# MiniMax H3 on Google Colab (ComfyUI 0.31.0)

Part of **AICU's ComfyLTS knowledge base** — a license-cleared, region-safe, long-term-support
ComfyUI base built and maintained by AICU Japan. This notebook documents how to evaluate the
**MiniMax H3** audio+video generation stack on Colab, pinned to a known-good ComfyUI line.

> **Status:** knowledge-base artifact. It is written to be runnable on Colab (best-effort,
> well-documented) but the timings and config below are verified from our own RTX 4000 Ada 20GB runs,
> not from a Colab execution of this exact notebook. Treat it as a reproducible recipe, not a guarantee.

---

## ⚠️ LICENSE / region note — read before running

**MiniMax H3 is distributed under the MiniMax H3 Community License, which is region-restricted.**
Per that community license the models are **not licensed for use in the EU, the UK, South Korea, or
the US**. This notebook is provided strictly for **evaluation** by eligible users.

- For **production serving**, AICU offers H3 through **ComfyPods**, where **region and age are gated
  by AICU auth** so the model is only offered to eligible users. Do not use this evaluation notebook
  to serve H3 to end users.
- You are responsible for confirming your own eligibility under the MiniMax H3 Community License
  before downloading or running the models below.

## GPU requirements — be honest, H3 is heavy

The full H3 stack is large. It loads, simultaneously:

- a **UNET / diffusion transformer** (the int8 build is ~21 GB on disk; the w4a8 build ~12.5 GB),
- a **video VAE** and a separate **audio VAE**, and
- a **large text encoder — Qwen3-VL (32B)** in NVFP4/AWQ.

What that means for Colab:

| Colab tier | GPU | H3 full stack? |
|---|---|---|
| Free | **T4 16 GB** | ❌ **Not enough.** The T4 cannot hold the full H3 stack (UNET + video VAE + Qwen3-VL). Do not expect the free tier to run this. |
| Pro | **A100 40 GB** | ✅ The quantized stack fits. This is the recommended target for the `int8` + `fp16` quality config. |
| Pro | L4 24 GB / others | ⚠️ Marginal — only with the lighter `w4a8` UNET and aggressive offload; expect slow or OOM. |

**Runtime / PyTorch:** ComfyUI 0.31.0 uses **DynamicVRAM**, which needs **PyTorch 2.8+**. Colab's
recent runtimes ship torch 2.8+ (some as new as 2.11), so a fresh Colab runtime is usually fine.
**Do not let `pip install -r requirements.txt` downgrade torch** — the next cell checks the version
and warns if it is below 2.8.

In [ ]:
# --- Check GPU and PyTorch version -----------------------------------------
!nvidia-smi

import torch
from packaging import version

print("\ntorch.__version__ =", torch.__version__)
print("CUDA available   =", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU              =", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM             = {total_gb:.1f} GB")
    if total_gb < 24:
        print("\n[WARN] <24 GB VRAM detected. The full MiniMax H3 stack likely will NOT fit.")
        print("       Free-tier T4 (16 GB) is not enough. Prefer Colab Pro A100 (40 GB).")

# ComfyUI 0.31.0 DynamicVRAM requires torch >= 2.8
if version.parse(torch.__version__.split('+')[0]) < version.parse("2.8"):
    print("\n[WARN] torch < 2.8 — ComfyUI 0.31.0 DynamicVRAM needs torch >= 2.8.")
    print("       Use a newer Colab runtime; do NOT let requirements.txt downgrade torch.")
else:
    print("\n[OK] torch >= 2.8 — DynamicVRAM supported.")

### Clone ComfyUI, pinned to v0.31.0

We pin to the ComfyLTS-blessed tag so frontend/backend/torch never drift. On Colab, torch is
**already installed** — we install ComfyUI's other requirements but must not clobber the existing torch.

In [ ]:
# --- Clone ComfyUI pinned to the LTS tag -----------------------------------
import os
if not os.path.isdir("ComfyUI"):
    !git clone --depth 1 --branch v0.31.0 https://github.com/comfyanonymous/ComfyUI
%cd ComfyUI
!git describe --tags --always

# Install ComfyUI's requirements. On Colab torch is already present and is a
# newer build than the runtime default; we do NOT want pip to downgrade it.
# Record the current torch, install, then verify it did not move.
_torch_before = None
try:
    import torch as _t
    _torch_before = _t.__version__
except Exception:
    pass

!pip install -q -r requirements.txt

import importlib, torch as _t2
importlib.reload(_t2)
print("torch after requirements install:", _t2.__version__)
if _torch_before and _t2.__version__ != _torch_before:
    print(f"[WARN] torch changed {_torch_before} -> {_t2.__version__}. "
          f"If it dropped below 2.8, reinstall the runtime's torch before continuing.")

### SageAttention ("H3 Sage")

SageAttention gives a **quality-preserving speedup** on the H3 attention path. We install it here and
launch ComfyUI with `--use-sage-attention` further down. This is part of our verified quality config.

In [ ]:
# --- SageAttention: quality-preserving speedup -----------------------------
!pip install -q sageattention
try:
    import sageattention
    print("sageattention installed:", getattr(sageattention, "__version__", "ok"))
except Exception as e:
    print("[WARN] sageattention import failed:", e)
    print("       You can still run without it; drop --use-sage-attention at launch.")

### Download the MiniMax H3 models from Hugging Face

We pull from **`Comfy-Org/MiniMax-H3`** into `ComfyUI/models/{diffusion_models,vae,text_encoders}`.

> **Note:** adjust the repo id if the org path differs — check
> [https://huggingface.co/Comfy-Org](https://huggingface.co/Comfy-Org). If any single filename
> 404s, browse the repo's file list and update the name; the rest of the notebook only depends on the
> local paths, not on the exact remote layout.

**UNET choice (pick one):**

| File | Size | Notes |
|---|---|---|
| `minimax_h3_fl2va_pruned_int8_convrot.safetensors` | ~21 GB | **int8, best quality** — needs A100 40 GB. Our quality config. |
| `minimax_h3_fl2va_pruned_w4a8_mixed.safetensors` | ~12.5 GB | w4a8, lighter/faster but **softer** output. Use if int8 won't fit. |

**VAE:** `minimax_h3_video_vae_fp16.safetensors` (full fp16 — **use this for quality**) and
`minimax_h3_audio_vae_fp32.safetensors`.

**Text encoder:** `qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors` (loaded via `CLIPLoader` with
`type="minimax"`).

If your HF account needs to accept the H3 license, log in first with `huggingface_hub.login()`.

In [ ]:
# --- Download H3 models -----------------------------------------------------
# Requires that your HF account is eligible under the MiniMax H3 Community License.
# If the repo is gated, run: from huggingface_hub import login; login()
from huggingface_hub import hf_hub_download
import os

REPO_ID = "Comfy-Org/MiniMax-H3"   # adjust if the org path differs (see https://huggingface.co/Comfy-Org)

# Pick the UNET build. int8 = best quality (needs A100 40GB); w4a8 = lighter/softer.
USE_INT8 = True   # set False on smaller GPUs to use the w4a8 build
UNET_FILE = ("minimax_h3_fl2va_pruned_int8_convrot.safetensors" if USE_INT8
             else "minimax_h3_fl2va_pruned_w4a8_mixed.safetensors")

DOWNLOADS = {
    "models/diffusion_models": [UNET_FILE],
    "models/vae": [
        "minimax_h3_video_vae_fp16.safetensors",   # use fp16 video VAE for quality
        "minimax_h3_audio_vae_fp32.safetensors",
    ],
    "models/text_encoders": [
        "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    ],
}

for subdir, files in DOWNLOADS.items():
    os.makedirs(subdir, exist_ok=True)
    for fname in files:
        print(f"downloading {fname} -> {subdir}/ ...")
        hf_hub_download(
            repo_id=REPO_ID,
            filename=fname,
            local_dir=subdir,
            local_dir_use_symlinks=False,
        )
        print(f"  done: {subdir}/{fname}")

print("\nUNET in use:", UNET_FILE)

## ⭐ Critical quality learning — 20 steps, NO turbo LoRA

This is the single most important finding from our runs, and it is baked into the workflow below:

- **Use 20 steps. Do NOT use a 4-step turbo LoRA.** We measured that the **4-step turbo is what
  causes the heavy blur** — the "dreamy veil" over the whole frame. The 20-step, no-turbo path is
  **sharp and usable**.
- **Step count dominates.** UNET precision (`w4a8` vs `int8`) and VAE precision (int8 vs fp16) had
  only a **minor** effect on the blur; the number of steps is what mattered.
- **Our quality config:** `int8` UNET **+** `fp16` video VAE **+** SageAttention **+** 20 steps
  (no turbo), sampler `res_multistep`, scheduler `simple`, `denoise 1.0`.
- **Resolution:** `864 x 480`.
- **Frame count rule:** H3 requires `length = 5 + 17n` (e.g. **56, 73, 90, 107, 124**). Other lengths
  will be rejected by the model.

The workflow dictionary in the next cells encodes exactly this. It is an **image-to-video (I2V)**
graph, which is the configuration we have verified end-to-end.

### T2V vs I2V

This notebook defaults to the **I2V** node `MiniMaxH3ImageToVideo` (clip, vae, prompt, width, height,
length, first_frame) driven by a `LoadImage` node — **this is the graph we KNOW works**.

For pure **text-to-video** there is a corresponding H3 text node (likely `MiniMaxH3TextToVideo`).
We have **not verified the exact class name / socket names** for the T2V node in this build, so it is
left as a note rather than the default. To try T2V: swap `MiniMaxH3ImageToVideo` for the T2V class,
drop the `first_frame` input (and the `LoadImage` node), and keep everything else — steps, sampler,
scheduler, resolution, and the frame-count rule — identical.

### Provide a first-frame image (for I2V)

`LoadImage` reads from `ComfyUI/input/`. Upload your own first frame, or let the cell generate a
simple placeholder so the notebook can run end-to-end without any manual step.

In [ ]:
# --- First-frame image into ComfyUI/input/ ---------------------------------
import os
os.makedirs("input", exist_ok=True)
INPUT_IMAGE = "first_frame.png"      # filename referenced by the LoadImage node
dst = os.path.join("input", INPUT_IMAGE)

uploaded = False
try:
    # On Colab: prompt for an upload. Comment this block out to always use the placeholder.
    from google.colab import files  # type: ignore
    print("Select a first-frame image to upload (or press Cancel to use a placeholder)...")
    got = files.upload()
    if got:
        src_name = next(iter(got))
        with open(dst, "wb") as f:
            f.write(got[src_name])
        uploaded = True
        print("uploaded ->", dst)
except Exception:
    pass

if not uploaded:
    # Placeholder: a simple gradient at the target resolution so LoadImage has something valid.
    from PIL import Image
    W, H = 864, 480
    img = Image.new("RGB", (W, H))
    px = img.load()
    for y in range(H):
        for x in range(W):
            px[x, y] = (int(255 * x / W), int(255 * y / H), 160)
    img.save(dst)
    print("wrote placeholder ->", dst, f"({W}x{H})")

### Launch ComfyUI (background) with SageAttention

We start ComfyUI listening on `127.0.0.1:8188` in the background and wait for it to come up, then
drive it over the HTTP API (`/prompt`, `/history`). `--lowvram` plays nicely with H3's DynamicVRAM
offload; on a roomy A100 you can drop it or use `--normalvram`.

In [ ]:
# --- Launch ComfyUI in the background --------------------------------------
import subprocess, time, os, urllib.request

os.makedirs("comfy_logs", exist_ok=True)
LOG = "comfy_logs/comfyui.log"

# main.py is launched from the ComfyUI dir (we %cd'd into it earlier).
_log = open(LOG, "w")
proc = subprocess.Popen(
    ["python", "main.py",
     "--listen", "127.0.0.1", "--port", "8188",
     "--lowvram", "--use-sage-attention"],
    stdout=_log, stderr=subprocess.STDOUT,
)
print("launched ComfyUI, pid", proc.pid, "-> logs in", LOG)

BASE = "http://127.0.0.1:8188"

def wait_for_server(timeout=600):
    start = time.time()
    while time.time() - start < timeout:
        if proc.poll() is not None:
            raise RuntimeError(f"ComfyUI exited early (code {proc.returncode}). See {LOG}.")
        try:
            with urllib.request.urlopen(BASE + "/system_stats", timeout=3) as r:
                if r.status == 200:
                    print(f"ComfyUI is up after {time.time()-start:.0f}s")
                    return True
        except Exception:
            pass
        time.sleep(3)
    raise TimeoutError(f"ComfyUI did not become ready within {timeout}s. See {LOG}.")

wait_for_server()

### Build the H3 I2V workflow (API format)

We build the graph as a Python dict (ComfyUI **API format**) rather than pasting a giant JSON blob.
This is the verified 20-step, no-turbo config: `UNETLoader` → `CLIPLoader(type="minimax")` →
two `VAELoader`s → `MiniMaxH3ImageToVideo` → `BasicGuider`/`KSamplerSelect`/`BasicScheduler`/
`RandomNoise` → `SamplerCustomAdvanced` → `VAEDecode` + `VAEDecodeAudio` → `CreateVideo` → `SaveVideo`.

In [ ]:
# --- Build the ComfyUI API-format workflow ---------------------------------
# Verified I2V graph: int8 UNET + fp16 video VAE + Sage + 20 steps (no turbo).

# ---- Editable parameters ----
PROMPT_TEXT = (
    "Chibi anime style. A cheerful character with green hair and orange accents smiles "
    "and waves hello at the camera, slight happy bounce and hair sway, warm friendly mood"
)
WIDTH  = 864       # verified resolution
HEIGHT = 480
LENGTH = 56        # MUST satisfy H3's rule: length = 5 + 17n  (56, 73, 90, 107, 124, ...)
STEPS  = 20        # 20-step, NO turbo LoRA. Do not drop to 4-step turbo (heavy blur).
SEED   = 20260810
FPS    = 24.0
FILENAME_PREFIX = "h3_colab"

assert (LENGTH - 5) % 17 == 0, "LENGTH must equal 5 + 17n (e.g. 56, 73, 90, 107, 124)"

def build_workflow():
    return {
        "1": {"class_type": "UNETLoader",
              "inputs": {"unet_name": UNET_FILE, "weight_dtype": "default"}},
        "2": {"class_type": "CLIPLoader",
              "inputs": {"clip_name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
                         "type": "minimax", "device": "default"}},
        "3": {"class_type": "VAELoader",
              "inputs": {"vae_name": "minimax_h3_video_vae_fp16.safetensors"}},
        "4": {"class_type": "VAELoader",
              "inputs": {"vae_name": "minimax_h3_audio_vae_fp32.safetensors"}},
        "5": {"class_type": "MiniMaxH3ImageToVideo",
              "inputs": {"clip": ["2", 0], "vae": ["3", 0],
                         "prompt": PROMPT_TEXT,
                         "width": WIDTH, "height": HEIGHT, "length": LENGTH,
                         "first_frame": ["15", 0]}},
        "6": {"class_type": "BasicGuider",
              "inputs": {"model": ["1", 0], "conditioning": ["5", 0]}},
        "7": {"class_type": "KSamplerSelect",
              "inputs": {"sampler_name": "res_multistep"}},
        "8": {"class_type": "BasicScheduler",
              "inputs": {"model": ["1", 0], "scheduler": "simple",
                         "steps": STEPS, "denoise": 1.0}},
        "9": {"class_type": "RandomNoise",
              "inputs": {"noise_seed": SEED}},
        "10": {"class_type": "SamplerCustomAdvanced",
               "inputs": {"noise": ["9", 0], "guider": ["6", 0],
                          "sampler": ["7", 0], "sigmas": ["8", 0],
                          "latent_image": ["5", 1]}},
        "11": {"class_type": "VAEDecode",
               "inputs": {"samples": ["10", 0], "vae": ["3", 0]}},
        "12": {"class_type": "VAEDecodeAudio",
               "inputs": {"samples": ["10", 0], "vae": ["4", 0]}},
        "13": {"class_type": "CreateVideo",
               "inputs": {"images": ["11", 0], "audio": ["12", 0],
                          "fps": FPS, "bit_depth": 8}},
        "14": {"class_type": "SaveVideo",
               "inputs": {"video": ["13", 0], "filename_prefix": FILENAME_PREFIX,
                          "format": "auto", "codec": "auto"}},
        "15": {"class_type": "LoadImage",
               "inputs": {"image": INPUT_IMAGE}},
    }

workflow = build_workflow()
print("workflow nodes:", len(workflow))
print("UNET:", workflow["1"]["inputs"]["unet_name"])
print(f"{WIDTH}x{HEIGHT} x {LENGTH} frames, {STEPS} steps (no turbo)")

In [ ]:
# --- Submit to ComfyUI and wait for the result -----------------------------
import json, uuid, time, urllib.request, urllib.error

CLIENT_ID = "h3-colab-" + uuid.uuid4().hex[:8]

def queue_prompt(wf):
    payload = json.dumps({"prompt": wf, "client_id": CLIENT_ID}).encode()
    req = urllib.request.Request(BASE + "/prompt", data=payload,
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req) as r:
            return json.loads(r.read())
    except urllib.error.HTTPError as e:
        raise RuntimeError("ComfyUI rejected the prompt:\n" + e.read().decode())

def get_history(pid):
    with urllib.request.urlopen(f"{BASE}/history/{pid}") as r:
        return json.loads(r.read())

resp = queue_prompt(workflow)
prompt_id = resp["prompt_id"]
print("queued prompt_id:", prompt_id)

# Poll history until this prompt_id appears (i.e. it finished).
# On RTX 4000 Ada 20GB, 864x480x124 @ 20 steps took ~560s; scale expectations to your GPU/length.
t0 = time.time()
result = None
while time.time() - t0 < 3600:
    if proc.poll() is not None:
        raise RuntimeError(f"ComfyUI process died (code {proc.returncode}). See {LOG}.")
    hist = get_history(prompt_id)
    if prompt_id in hist:
        result = hist[prompt_id]
        break
    print(f"  ...running {time.time()-t0:.0f}s", end="\r")
    time.sleep(5)

if result is None:
    raise TimeoutError("Generation did not finish within the timeout window.")
print(f"\nfinished in {time.time()-t0:.0f}s")

# Locate the saved video in the outputs.
output_video = None
for node_id, node_out in result.get("outputs", {}).items():
    for key in ("videos", "gifs", "images"):
        for item in node_out.get(key, []):
            fn = item.get("filename", "")
            if fn.lower().endswith((".mp4", ".webm", ".mov")):
                sub = item.get("subfolder", "")
                output_video = os.path.join("output", sub, fn)
    if output_video:
        break

print("output video:", output_video)

## Timing (measured)

On **RTX 4000 Ada 20 GB** (our reference, not Colab):

| Config | Wall time | Quality |
|---|---|---|
| Full **5.17s** clip (864×480 × **124** frames), **20-step no-turbo** | **≈ 560 s (~9 min)** | sharp / usable — our quality config |
| 4-step **turbo** path | ~140–170 s | **blurry — draft only** ("dreamy veil") |

An **A100 (Colab Pro)** should be faster than the RTX 4000 Ada figures. Scale the poll timeout to
your GPU and chosen `LENGTH`. The turbo path is only for quick previews — never ship it.

### Display the result inline

In [ ]:
# --- Show the generated mp4 inline -----------------------------------------
from IPython.display import Video, HTML, display
import os, base64, mimetypes

if output_video and os.path.exists(output_video):
    print("playing:", output_video, f"({os.path.getsize(output_video)/1024**2:.1f} MB)")
    try:
        display(Video(output_video, embed=True, width=WIDTH, height=HEIGHT))
    except Exception:
        # Fallback: embed as a base64 data URL in an HTML5 <video> tag.
        mime = mimetypes.guess_type(output_video)[0] or "video/mp4"
        b64 = base64.b64encode(open(output_video, "rb").read()).decode()
        display(HTML(f'<video controls width="{WIDTH}" '
                     f'src="data:{mime};base64,{b64}"></video>'))
else:
    print("No output video found. Check the ComfyUI log:", LOG)
    !tail -n 40 {LOG}